In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
 

In [2]:
pd.set_option('display.width', 140)
 
TIME_SLOTS = {
    'night_late': {'start': '00:00', 'end': '06:00', 'id': 0},
    'early_morning': {'start': '06:00', 'end': '09:00', 'id': 1},
    'work_morning': {'start': '09:00', 'end': '12:00', 'id': 2},
    'lunch_break': {'start': '12:00', 'end': '15:00', 'id': 3},
    'work_afternoon': {'start': '15:00', 'end': '18:00', 'id': 4},
    'evening_prime': {'start': '18:00', 'end': '21:00', 'id': 5},
    'night_wind': {'start': '21:00', 'end': '23:59', 'id': 6}
}
 
def assign_time_slot(timestamp):
    if isinstance(timestamp, str):
        timestamp = pd.to_datetime(timestamp)
    ts_time = timestamp.time()
    for slot_name, slot_info in TIME_SLOTS.items():
        start_time = datetime.strptime(slot_info['start'], '%H:%M').time()
        end_time = datetime.strptime(slot_info['end'], '%H:%M').time()
        if slot_info['id'] < 6:
            if start_time <= ts_time < end_time:
                return slot_info['id']
        else:
            if start_time <= ts_time <= end_time:
                return slot_info['id']
    return -1

In [7]:
# 1. Charger les données préparées
# ----------------------------------------------------------------
df = pd.read_csv("../data/prepared/resultPreparation.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"[1] Données chargées: {df.shape}")

[1] Données chargées: (39371, 25)


In [10]:
# 2. Sélection des features et de la cible
# ----------------------------------------------------------------
NUMERIC_FEATURES = ['hour', 'day_of_week', 'month', 'message_length',
                     'has_hashtag', 'has_link', 'has_emoji']
BOOL_FEATURES = ['is_weekend', 'is_business_hours']
CATEGORICAL_FEATURES = ['entreprise', 'content_categorie', 'time_slot_id']
 
FEATURES = NUMERIC_FEATURES + BOOL_FEATURES + CATEGORICAL_FEATURES
TARGET = 'total_engagement'
 
model_df = df[FEATURES + [TARGET, 'timestamp']].copy()
model_df['year'] = pd.to_datetime(model_df['timestamp']).dt.year
model_df = model_df.dropna(subset=FEATURES + [TARGET])
for c in BOOL_FEATURES:
    model_df[c] = model_df[c].astype(int)
model_df['content_categorie'] = model_df['content_categorie'].fillna('unknown')
 
print(f"[2] Après nettoyage features: {model_df.shape}")

# Normalisation : engagement relatif à la baseline (médiane) de la page pour son année
# -> isole l'effet du moment de publication, indépendamment de la dérive temporelle
# (chute d'engagement générale observée en 2022 sur les 3 pages)
baseline = model_df.groupby(['entreprise', 'year'])[TARGET].transform('median')
baseline = baseline.replace(0, np.nan)
model_df['engagement_relative'] = model_df[TARGET] / baseline
model_df = model_df.dropna(subset=['engagement_relative'])
 
# Transformation log1p de la cible relative (toujours utile : distribution asymétrique)
model_df['target_log'] = np.log1p(model_df['engagement_relative'])
model_df['baseline_used'] = baseline.loc[model_df.index]
 


[2] Après nettoyage features: (38864, 15)


In [11]:
# 3. Split ALÉATOIRE, stratifié par entreprise (indépendant de l'ancienneté)
# ----------------------------------------------------------------
from sklearn.model_selection import train_test_split
 
train_df, test_df = train_test_split(
    model_df, test_size=0.2, random_state=42, stratify=model_df['entreprise']
)
print(f"[3] Train: {train_df.shape}, Test: {test_df.shape}")
print("    Répartition par entreprise (train / test):")
print(pd.concat([
    train_df['entreprise'].value_counts().rename('train'),
    test_df['entreprise'].value_counts().rename('test')
], axis=1))
 
X_train = train_df[FEATURES]
X_test = test_df[FEATURES]
y_train_log = train_df['target_log']
y_test_log = test_df['target_log']
y_test_raw = test_df['engagement_relative']

[3] Train: (31091, 18), Test: (7773, 18)
    Répartition par entreprise (train / test):
            train  test
entreprise             
Orange      14272  3568
TT           9690  2423
Ooredoo      7129  1782


In [12]:
# 4. Préprocesseur (one-hot pour les catégorielles)
# ----------------------------------------------------------------
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)
], remainder='passthrough')

In [14]:
# 5. Modèle 1 : Random Forest
# ----------------------------------------------------------------
rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=5,
                                     random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train_log)
rf_pred_log = rf_pipeline.predict(X_test)
rf_pred_raw = np.expm1(rf_pred_log)
rf_pred_raw = np.clip(rf_pred_raw, 0, None)  # pas d'engagement négatif

In [15]:
# 6. Modèle 2 : XGBoost
# ----------------------------------------------------------------
xgb_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8,
                                random_state=42, n_jobs=-1))
])
xgb_pipeline.fit(X_train, y_train_log)
xgb_pred_log = xgb_pipeline.predict(X_test)
xgb_pred_raw = np.expm1(xgb_pred_log)
xgb_pred_raw = np.clip(xgb_pred_raw, 0, None)

In [16]:
# 7. Évaluation comparative
# ----------------------------------------------------------------
def evaluate(name, y_true_log, y_pred_log, y_true_raw, y_pred_raw):
    print(f"\n--- {name} ---")
    print(f"  [Échelle log]   RMSE: {np.sqrt(mean_squared_error(y_true_log, y_pred_log)):.3f} | "
          f"MAE: {mean_absolute_error(y_true_log, y_pred_log):.3f} | "
          f"R²: {r2_score(y_true_log, y_pred_log):.3f}")
    print(f"  [Échelle brute] RMSE: {np.sqrt(mean_squared_error(y_true_raw, y_pred_raw)):.1f} | "
          f"MAE: {mean_absolute_error(y_true_raw, y_pred_raw):.1f} | "
          f"R²: {r2_score(y_true_raw, y_pred_raw):.3f}")
 
evaluate("Random Forest", y_test_log, rf_pred_log, y_test_raw, rf_pred_raw)
evaluate("XGBoost", y_test_log, xgb_pred_log, y_test_raw, xgb_pred_raw)


--- Random Forest ---
  [Échelle log]   RMSE: 0.781 | MAE: 0.539 | R²: 0.155
  [Échelle brute] RMSE: 24.1 | MAE: 3.9 | R²: 0.300

--- XGBoost ---
  [Échelle log]   RMSE: 0.781 | MAE: 0.542 | R²: 0.153
  [Échelle brute] RMSE: 21.8 | MAE: 3.8 | R²: 0.429


In [17]:
# 8. Importance des features (XGBoost)
# ----------------------------------------------------------------
feature_names = xgb_pipeline.named_steps['prep'].get_feature_names_out()
importances = xgb_pipeline.named_steps['model'].feature_importances_
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values(
    'importance', ascending=False).head(15)
print("\n--- Top 15 features importantes (XGBoost) ---")
print(imp_df.to_string(index=False))
 
print("\n✓ Test modélisation terminé sans erreur")
 


--- Top 15 features importantes (XGBoost) ---
                                    feature  importance
        cat__content_categorie_storie_photo    0.253370
        cat__content_categorie_storie_video    0.135312
        cat__content_categorie_shared_story    0.059707
         cat__content_categorie_added_video    0.046528
                     cat__entreprise_Orange    0.045985
                        remainder__has_link    0.040822
               remainder__is_business_hours    0.037347
                        cat__time_slot_id_5    0.035975
                  remainder__message_length    0.034276
        cat__content_categorie_added_photos    0.032133
cat__content_categorie_mobile_status_update    0.029228
                         cat__entreprise_TT    0.026315
                            remainder__hour    0.025796
                           remainder__month    0.024563
                    cat__entreprise_Ooredoo    0.024057

✓ Test modélisation terminé sans erreur


In [18]:
# 9. Génération du pattern de publication optimal
# ------------------------------------------------------------------
# On utilise XGBoost (meilleur R² sur l'échelle brute)
BEST_MODEL = xgb_pipeline
 
# Valeurs "neutres" pour les features non liées au moment/type de post
# (médiane pour numériques, mode pour booléens) -> isole l'effet jour/heure/contenu
message_length_default = model_df['message_length'].median()
has_hashtag_default = int(model_df['has_hashtag'].mode()[0])
has_link_default = int(model_df['has_link'].mode()[0])
has_emoji_default = int(model_df['has_emoji'].mode()[0])
 
entreprises_list = model_df['entreprise'].unique()
content_types = model_df['content_categorie'].unique()
hours = range(24)
days = range(7)
months = range(1, 13)  # on moyenne sur les mois pour neutraliser la saisonnalité
 
rows = []
for ent in entreprises_list:
    for day in days:
        for hour in hours:
            for content in content_types:
                slot_id = assign_time_slot(pd.Timestamp(year=2024, month=1, day=1, hour=hour))
                is_weekend = day in [5, 6]
                is_business_hours = 8 < hour < 18
                for month in months:
                    rows.append({
                        'entreprise': ent, 'day_of_week': day, 'hour': hour,
                        'content_categorie': content, 'month': month,
                        'message_length': message_length_default,
                        'has_hashtag': has_hashtag_default, 'has_link': has_link_default,
                        'has_emoji': has_emoji_default, 'is_weekend': is_weekend,
                        'is_business_hours': is_business_hours, 'time_slot_id': slot_id
                    })
 
grid_df = pd.DataFrame(rows)
print(f"\n[9] Grille de combinaisons générée: {grid_df.shape}")
 
grid_df['pred_log'] = BEST_MODEL.predict(grid_df[FEATURES])
grid_df['pred_relative'] = np.expm1(grid_df['pred_log'])
 
# Moyenne sur les mois -> pattern jour/heure/contenu indépendant de la saisonnalité
pattern = grid_df.groupby(['entreprise', 'day_of_week', 'hour', 'content_categorie'])['pred_relative'].mean().reset_index()
 
# Garde-fou niveau 1 : type de contenu suffisamment représenté globalement
MIN_POSTS_THRESHOLD = 30
content_counts = model_df.groupby(['entreprise', 'content_categorie']).size().reset_index(name='n_posts')
valid_combos = content_counts[content_counts['n_posts'] >= MIN_POSTS_THRESHOLD][['entreprise', 'content_categorie']]
 
print(f"\n[9b] Combinaisons entreprise x type de contenu écartées (< {MIN_POSTS_THRESHOLD} posts historiques):")
print(content_counts[content_counts['n_posts'] < MIN_POSTS_THRESHOLD].to_string(index=False))
 
pattern = pattern.merge(valid_combos, on=['entreprise', 'content_categorie'], how='inner')
 
# Garde-fou niveau 2 (le plus important) : la combinaison précise
# entreprise x jour x créneau horaire x type de contenu doit avoir un minimum
# d'historique réel, sinon on regroupe au niveau créneau (7 tranches) plutôt
# que l'heure exacte -> évite qu'un post isolé/viral fausse la recommandation
# (cas vécu: 1 seul post à 3h du matin avec un engagement exceptionnel a suffi
# à faire recommander ce créneau, qui n'a aucune valeur statistique)
MIN_SLOT_SUPPORT = 5
 
model_df['time_slot_id_hist'] = model_df['time_slot_id']  # déjà calculé dans le pipeline
support = model_df.groupby(
    ['entreprise', 'content_categorie', 'day_of_week', 'time_slot_id_hist']
).size().reset_index(name='n_posts_slot')
 
# on rattache le créneau (time_slot_id) à chaque ligne de la grille horaire
grid_hour_to_slot = {h: assign_time_slot(pd.Timestamp(year=2024, month=1, day=1, hour=h)) for h in range(24)}
pattern['time_slot_id'] = pattern['hour'].map(grid_hour_to_slot)
 
pattern = pattern.merge(
    support.rename(columns={'time_slot_id_hist': 'time_slot_id'}),
    on=['entreprise', 'content_categorie', 'day_of_week', 'time_slot_id'],
    how='left'
)
pattern['n_posts_slot'] = pattern['n_posts_slot'].fillna(0)
 
n_before = len(pattern)
pattern_reliable = pattern[pattern['n_posts_slot'] >= MIN_SLOT_SUPPORT].copy()
print(f"\n[9c] Combinaisons jour x créneau x contenu écartées "
      f"(< {MIN_SLOT_SUPPORT} posts historiques dans ce créneau précis): "
      f"{n_before - len(pattern_reliable)} / {n_before}")
 
pattern = pattern_reliable
 
day_names_fr = {0: 'Lundi', 1: 'Mardi', 2: 'Mercredi', 3: 'Jeudi', 4: 'Vendredi', 5: 'Samedi', 6: 'Dimanche'}
pattern['jour'] = pattern['day_of_week'].map(day_names_fr)
 
print("\n=== TOP 5 CRÉNEAUX RECOMMANDÉS PAR ENTREPRISE ===")
for ent in entreprises_list:
    sub = pattern[pattern['entreprise'] == ent].sort_values('pred_relative', ascending=False).head(5)
    print(f"\n🏢 {ent}:")
    for _, r in sub.iterrows():
        print(f"  {r['jour']} à {int(r['hour'])}h — {r['content_categorie']} "
              f"(score relatif prédit: {r['pred_relative']:.2f}x la baseline, "
              f"basé sur {int(r['n_posts_slot'])} posts historiques)")
 
pattern.to_csv("optimal_pattern_test.csv", index=False)
print("\n✓ Pattern optimal exporté")
 
 


[9] Grille de combinaisons générée: (48384, 12)

[9b] Combinaisons entreprise x type de contenu écartées (< 30 posts historiques):
entreprise content_categorie  n_posts
   Ooredoo   published_story        4
    Orange     created_event        2
    Orange   published_story        1

[9c] Combinaisons jour x créneau x contenu écartées (< 5 posts historiques dans ce créneau précis): 801 / 2352

=== TOP 5 CRÉNEAUX RECOMMANDÉS PAR ENTREPRISE ===

🏢 Ooredoo:
  Vendredi à 20h — mobile_status_update (score relatif prédit: 3.10x la baseline, basé sur 15 posts historiques)
  Vendredi à 19h — mobile_status_update (score relatif prédit: 3.06x la baseline, basé sur 15 posts historiques)
  Lundi à 20h — mobile_status_update (score relatif prédit: 2.94x la baseline, basé sur 14 posts historiques)
  Samedi à 20h — mobile_status_update (score relatif prédit: 2.87x la baseline, basé sur 6 posts historiques)
  Mardi à 19h — mobile_status_update (score relatif prédit: 2.85x la baseline, basé sur 8 posts

In [25]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
 
pd.set_option('display.width', 140)
 
TIME_SLOTS = {
    'night_late': {'start': '00:00', 'end': '06:00', 'id': 0},
    'early_morning': {'start': '06:00', 'end': '09:00', 'id': 1},
    'work_morning': {'start': '09:00', 'end': '12:00', 'id': 2},
    'lunch_break': {'start': '12:00', 'end': '15:00', 'id': 3},
    'work_afternoon': {'start': '15:00', 'end': '18:00', 'id': 4},
    'evening_prime': {'start': '18:00', 'end': '21:00', 'id': 5},
    'night_wind': {'start': '21:00', 'end': '23:59', 'id': 6}
}
 
def assign_time_slot(timestamp):
    if isinstance(timestamp, str):
        timestamp = pd.to_datetime(timestamp)
    ts_time = timestamp.time()
    for slot_name, slot_info in TIME_SLOTS.items():
        start_time = datetime.strptime(slot_info['start'], '%H:%M').time()
        end_time = datetime.strptime(slot_info['end'], '%H:%M').time()
        if slot_info['id'] < 6:
            if start_time <= ts_time < end_time:
                return slot_info['id']
        else:
            if start_time <= ts_time <= end_time:
                return slot_info['id']
    return -1
 
# ----------------------------------------------------------------
# 1. Charger les données préparées
# ----------------------------------------------------------------
df = pd.read_csv("../data/prepared/resultPreparation.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"[1] Données chargées: {df.shape}")
 
# ----------------------------------------------------------------
# 2. Sélection des features et de la cible
# ----------------------------------------------------------------
NUMERIC_FEATURES = ['hour', 'day_of_week', 'month', 'message_length',
                     'has_hashtag', 'has_link', 'has_emoji']
BOOL_FEATURES = ['is_weekend', 'is_business_hours']
CATEGORICAL_FEATURES = ['entreprise', 'content_categorie', 'time_slot_id']
 
FEATURES = NUMERIC_FEATURES + BOOL_FEATURES + CATEGORICAL_FEATURES
TARGET = 'total_engagement'
 
model_df = df[FEATURES + [TARGET, 'timestamp']].copy()
model_df['year'] = pd.to_datetime(model_df['timestamp']).dt.year
model_df = model_df.dropna(subset=FEATURES + [TARGET])
for c in BOOL_FEATURES:
    model_df[c] = model_df[c].astype(int)
model_df['content_categorie'] = model_df['content_categorie'].fillna('unknown')
 
print(f"[2] Après nettoyage features: {model_df.shape}")
 
# Normalisation : engagement relatif à la baseline (médiane) de la page pour son année
# -> isole l'effet du moment de publication, indépendamment de la dérive temporelle
# (chute d'engagement générale observée en 2022 sur les 3 pages)
baseline = model_df.groupby(['entreprise', 'year'])[TARGET].transform('median')
baseline = baseline.replace(0, np.nan)
model_df['engagement_relative'] = model_df[TARGET] / baseline
model_df = model_df.dropna(subset=['engagement_relative'])
 
# Transformation log1p de la cible relative (toujours utile : distribution asymétrique)
model_df['target_log'] = np.log1p(model_df['engagement_relative'])
model_df['baseline_used'] = baseline.loc[model_df.index]
 
# ----------------------------------------------------------------
# 3. Split ALÉATOIRE, stratifié par entreprise (indépendant de l'ancienneté)
# ----------------------------------------------------------------
from sklearn.model_selection import train_test_split
 
train_df, test_df = train_test_split(
    model_df, test_size=0.2, random_state=42, stratify=model_df['entreprise']
)
print(f"[3] Train: {train_df.shape}, Test: {test_df.shape}")
print("    Répartition par entreprise (train / test):")
print(pd.concat([
    train_df['entreprise'].value_counts().rename('train'),
    test_df['entreprise'].value_counts().rename('test')
], axis=1))
 
X_train = train_df[FEATURES]
X_test = test_df[FEATURES]
y_train_log = train_df['target_log']
y_test_log = test_df['target_log']
y_test_raw = test_df['engagement_relative']
 
# ----------------------------------------------------------------
# 4. Préprocesseur (one-hot pour les catégorielles)
# ----------------------------------------------------------------
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)
], remainder='passthrough')
 
# ----------------------------------------------------------------
# 5. Modèle 1 : Random Forest
# ----------------------------------------------------------------
rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=5,
                                     random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train_log)
rf_pred_log = rf_pipeline.predict(X_test)
rf_pred_raw = np.expm1(rf_pred_log)
rf_pred_raw = np.clip(rf_pred_raw, 0, None)  # pas d'engagement négatif
 
# ----------------------------------------------------------------
# 6. Modèle 2 : XGBoost
# ----------------------------------------------------------------
xgb_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8,
                                random_state=42, n_jobs=-1))
])
xgb_pipeline.fit(X_train, y_train_log)
xgb_pred_log = xgb_pipeline.predict(X_test)
xgb_pred_raw = np.expm1(xgb_pred_log)
xgb_pred_raw = np.clip(xgb_pred_raw, 0, None)
 
# ----------------------------------------------------------------
# 7. Évaluation comparative
# ----------------------------------------------------------------
def evaluate(name, y_true_log, y_pred_log, y_true_raw, y_pred_raw):
    print(f"\n--- {name} ---")
    print(f"  [Échelle log]   RMSE: {np.sqrt(mean_squared_error(y_true_log, y_pred_log)):.3f} | "
          f"MAE: {mean_absolute_error(y_true_log, y_pred_log):.3f} | "
          f"R²: {r2_score(y_true_log, y_pred_log):.3f}")
    print(f"  [Échelle brute] RMSE: {np.sqrt(mean_squared_error(y_true_raw, y_pred_raw)):.1f} | "
          f"MAE: {mean_absolute_error(y_true_raw, y_pred_raw):.1f} | "
          f"R²: {r2_score(y_true_raw, y_pred_raw):.3f}")
 
evaluate("Random Forest", y_test_log, rf_pred_log, y_test_raw, rf_pred_raw)
evaluate("XGBoost", y_test_log, xgb_pred_log, y_test_raw, xgb_pred_raw)
 
# ----------------------------------------------------------------
# 8. Importance des features (XGBoost)
# ----------------------------------------------------------------
feature_names = xgb_pipeline.named_steps['prep'].get_feature_names_out()
importances = xgb_pipeline.named_steps['model'].feature_importances_
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values(
    'importance', ascending=False).head(15)
print("\n--- Top 15 features importantes (XGBoost) ---")
print(imp_df.to_string(index=False))
 
print("\n✓ Test modélisation terminé sans erreur")
 
# ------------------------------------------------------------------
# 9. Génération du pattern de publication optimal
# ------------------------------------------------------------------
# On utilise XGBoost (meilleur R² sur l'échelle brute)
BEST_MODEL = xgb_pipeline
 
# Valeurs "neutres" pour les features non liées au moment/type de post
# (médiane pour numériques, mode pour booléens) -> isole l'effet jour/heure/contenu
message_length_default = model_df['message_length'].median()
has_hashtag_default = int(model_df['has_hashtag'].mode()[0])
has_link_default = int(model_df['has_link'].mode()[0])
has_emoji_default = int(model_df['has_emoji'].mode()[0])
 
entreprises_list = model_df['entreprise'].unique()
content_types = model_df['content_categorie'].unique()
hours = range(24)
days = range(7)
months = range(1, 13)  # on moyenne sur les mois pour neutraliser la saisonnalité
 
rows = []
for ent in entreprises_list:
    for day in days:
        for hour in hours:
            for content in content_types:
                slot_id = assign_time_slot(pd.Timestamp(year=2024, month=1, day=1, hour=hour))
                is_weekend = day in [5, 6]
                is_business_hours = 8 < hour < 18
                for month in months:
                    rows.append({
                        'entreprise': ent, 'day_of_week': day, 'hour': hour,
                        'content_categorie': content, 'month': month,
                        'message_length': message_length_default,
                        'has_hashtag': has_hashtag_default, 'has_link': has_link_default,
                        'has_emoji': has_emoji_default, 'is_weekend': is_weekend,
                        'is_business_hours': is_business_hours, 'time_slot_id': slot_id
                    })
 
grid_df = pd.DataFrame(rows)
print(f"\n[9] Grille de combinaisons générée: {grid_df.shape}")
 
grid_df['pred_log'] = BEST_MODEL.predict(grid_df[FEATURES])
grid_df['pred_relative'] = np.expm1(grid_df['pred_log'])
 
# Moyenne sur les mois -> pattern jour/heure/contenu indépendant de la saisonnalité
pattern = grid_df.groupby(['entreprise', 'day_of_week', 'hour', 'content_categorie'])['pred_relative'].mean().reset_index()
 
# Garde-fou niveau 1 : type de contenu suffisamment représenté globalement
MIN_POSTS_THRESHOLD = 30
content_counts = model_df.groupby(['entreprise', 'content_categorie']).size().reset_index(name='n_posts')
valid_combos = content_counts[content_counts['n_posts'] >= MIN_POSTS_THRESHOLD][['entreprise', 'content_categorie']]
 
print(f"\n[9b] Combinaisons entreprise x type de contenu écartées (< {MIN_POSTS_THRESHOLD} posts historiques):")
print(content_counts[content_counts['n_posts'] < MIN_POSTS_THRESHOLD].to_string(index=False))
 
pattern = pattern.merge(valid_combos, on=['entreprise', 'content_categorie'], how='inner')
 
# Garde-fou niveau 2 (le plus important) : la combinaison précise
# entreprise x jour x créneau horaire x type de contenu doit avoir un minimum
# d'historique réel, sinon on regroupe au niveau créneau (7 tranches) plutôt
# que l'heure exacte -> évite qu'un post isolé/viral fausse la recommandation
# (cas vécu: 1 seul post à 3h du matin avec un engagement exceptionnel a suffi
# à faire recommander ce créneau, qui n'a aucune valeur statistique)
MIN_SLOT_SUPPORT = 5
 
model_df['time_slot_id_hist'] = model_df['time_slot_id']  # déjà calculé dans le pipeline
support = model_df.groupby(
    ['entreprise', 'content_categorie', 'day_of_week', 'time_slot_id_hist']
).size().reset_index(name='n_posts_slot')
 
# on rattache le créneau (time_slot_id) à chaque ligne de la grille horaire
grid_hour_to_slot = {h: assign_time_slot(pd.Timestamp(year=2024, month=1, day=1, hour=h)) for h in range(24)}
pattern['time_slot_id'] = pattern['hour'].map(grid_hour_to_slot)
 
pattern = pattern.merge(
    support.rename(columns={'time_slot_id_hist': 'time_slot_id'}),
    on=['entreprise', 'content_categorie', 'day_of_week', 'time_slot_id'],
    how='left'
)
pattern['n_posts_slot'] = pattern['n_posts_slot'].fillna(0)
 
n_before = len(pattern)
pattern_reliable = pattern[pattern['n_posts_slot'] >= MIN_SLOT_SUPPORT].copy()
print(f"\n[9c] Combinaisons jour x créneau x contenu écartées "
      f"(< {MIN_SLOT_SUPPORT} posts historiques dans ce créneau précis): "
      f"{n_before - len(pattern_reliable)} / {n_before}")
 
pattern = pattern_reliable
 
day_names_fr = {0: 'Lundi', 1: 'Mardi', 2: 'Mercredi', 3: 'Jeudi', 4: 'Vendredi', 5: 'Samedi', 6: 'Dimanche'}
pattern['jour'] = pattern['day_of_week'].map(day_names_fr)
 
print("\n=== TOP 5 CRÉNEAUX RECOMMANDÉS PAR ENTREPRISE ===")
for ent in entreprises_list:
    sub = pattern[pattern['entreprise'] == ent].sort_values('pred_relative', ascending=False).head(5)
    print(f"\n {ent}:")
    for _, r in sub.iterrows():
        print(f"  {r['jour']} à {int(r['hour'])}h — {r['content_categorie']} "
              f"(score relatif prédit: {r['pred_relative']:.2f}x la baseline, "
              f"basé sur {int(r['n_posts_slot'])} posts historiques)")
 
pattern.to_csv("optimal_pattern_test.csv", index=False)

 
 


[1] Données chargées: (39371, 25)
[2] Après nettoyage features: (38864, 15)
[3] Train: (31091, 18), Test: (7773, 18)
    Répartition par entreprise (train / test):
            train  test
entreprise             
Orange      14272  3568
TT           9690  2423
Ooredoo      7129  1782

--- Random Forest ---
  [Échelle log]   RMSE: 0.781 | MAE: 0.539 | R²: 0.155
  [Échelle brute] RMSE: 24.1 | MAE: 3.9 | R²: 0.300

--- XGBoost ---
  [Échelle log]   RMSE: 0.781 | MAE: 0.542 | R²: 0.153
  [Échelle brute] RMSE: 21.8 | MAE: 3.8 | R²: 0.429

--- Top 15 features importantes (XGBoost) ---
                                    feature  importance
        cat__content_categorie_storie_photo    0.253370
        cat__content_categorie_storie_video    0.135312
        cat__content_categorie_shared_story    0.059707
         cat__content_categorie_added_video    0.046528
                     cat__entreprise_Orange    0.045985
                        remainder__has_link    0.040822
               remainde